# 09 Using Custom Operators in a Workload

This example answers the question: once you write a Triton kernel, how do you plug it into a pLLM workload and verify with pLLM's checking tools that it neither breaks CUDA graphs nor disagrees with its CPU reference?

New inference-time algorithms increasingly rely on operations that cannot be expressed cleanly in a single line of PyTorch—or run too slowly if you do—such as table-lookup hashing, sparse-attention page scoring, or chunked scans. Researchers naturally reach for a custom Triton kernel. The catch when bringing that kernel into an inference engine is that it must stay inside CUDA graphs, must never sneak in a GPU-to-CPU synchronization, and needs a CPU reference implementation—otherwise, when something goes wrong, it is hard to tell whether the bug is in the kernel or elsewhere.

In pLLM, there is no new operator-wrapping API to learn: wrap your kernel with PyTorch's `torch.library.custom_op` and call it inside workload methods like any normal function. Once wired in, pLLM automatically does three things:
- Pre-launch `pllm.explain` lists the operator (per-row FLOPs, logical bytes, and whether a CPU reference exists).
- `pllm.testing.check` runs scheduling contract checks on CPU using your CPU reference.
- `pllm.testing.check_on_gpu` replays CPU inputs on the GPU and compares your kernel's outputs against the CPU reference (*device equivalence*).

**Example Algorithm Used Here**

In Example 06, we saw that the residual stream norm grows with layer depth, so a fixed steering vector has a different relative weight at different layers and tokens. Here we implement "relative steering": after layer 12's MLP, rewrite the residual stream $h$ as $h + \text{strength} \times |h| \times v$, where $v$ is a unit direction vector, scaling the intervention proportionally to each row's own norm. Computing a row-wise norm and adding back fits a "one program per row" Triton kernel naturally.

Before we start, note two facts:
- This kernel is no faster than PyTorch here: measured throughput on 7B is `98%` of native vLLM for both (see Step 8), well within measurement noise—a single addition at layer 12 is not the bottleneck of full-model inference. You write a custom op when your algorithm genuinely needs a step PyTorch cannot express or runs too slowly; this notebook focuses solely on wiring and verification.
- The steering direction is computed via CAA from 5 phrase pairs; a change under non-zero strength only shows the modification took effect, not that the model got better.

**Steps in This Notebook**

We walk through 10 steps:
1. Write the Triton kernel and wrap it with `custom_op`;
2. Compare GPU output against the CPU reference standalone;
3. Prepare the steering direction;
4. Write the workload;
5. Inspect operator info via `explain`;
6. Run CPU contract checks;
7. Run generation to observe steering behavior;
8. Benchmark throughput;
9. Run GPU contract checks (device equivalence);
10. Deliberately introduce a bug in the CPU reference and watch it get caught.

Let's import the libraries and configure `ModelConfig` for `Qwen/Qwen2.5-0.5B-Instruct`:

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors and suppress startup logs

import math
import tempfile
from dataclasses import dataclass

import torch
import triton
import triton.language as tl
from safetensors.torch import save_file
from torch.utils.flop_counter import register_flop_formula
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing
from pllm.benchmark import throughput

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)

## Step 1: Write the Kernel and Wrap It with `custom_op`

Here we solve: how do we wrap a Triton kernel into an operator recognized by both PyTorch and pLLM?

In `_relative_steer_kernel` below, each program processes one row: it loads `x` and `y` and adds them in `float32` to form `h`, loads the direction vector `v`, computes the row norm $|h|$, and stores $h + \text{strength} \cdot |h| \cdot v$.

Next, we wrap it with `@torch.library.custom_op("tutorial::relative_steer", mutates_args=(), device_types="cuda")`. This is standard PyTorch machinery—the same mechanism `torch.compile` uses to recognize custom operators.
Beneath it are three registrations, each serving a distinct purpose:
- `register_fake`: returns only the output shape/dtype without running real math. pLLM uses this for shape-level memory planning and sync checks; this registration is mandatory.
- `register_kernel("cpu")`: provides the CPU reference implementation, used by both CPU contract checks and GPU device-equivalence checks.
- `register_flop_formula`: tells `explain` how many FLOPs each call performs (counted here as 5 floating-point ops per element).

A few rules apply when using custom ops in pLLM:
- The kernel must return a new tensor (`mutates_args=()`), never mutate inputs in place; state updates must go through pLLM effects.
- `register_fake` is required.
- Inside workload methods, call the wrapped op object directly (`relative_steer` here) rather than `torch.ops.tutorial.relative_steer`, because pLLM needs to inspect its source.
- If training backpropagates through the op, add `register_autograd`; if used inside a per-token custom forward body, add `register_vmap`. `register_kernel("cpu")` and `register_flop_formula` are optional, though without a CPU implementation the CPU contract check cannot execute that math.

In [2]:
@triton.jit
def _relative_steer_kernel(x_ptr, y_ptr, v_ptr, out_ptr, strength, width, BLOCK: tl.constexpr):
    row = tl.program_id(0)  # One program processes one row
    cols = tl.arange(0, BLOCK)
    mask = cols < width
    h = tl.load(x_ptr + row * width + cols, mask, other=0.0).to(tl.float32)
    h += tl.load(y_ptr + row * width + cols, mask, other=0.0).to(tl.float32)
    v = tl.load(v_ptr + cols, mask, other=0.0).to(tl.float32)
    norm = tl.sqrt(tl.sum(h * h, axis=0))
    tl.store(out_ptr + row * width + cols, (h + strength * norm * v).to(out_ptr.dtype.element_ty), mask)


@torch.library.custom_op("tutorial::relative_steer", mutates_args=(), device_types="cuda")
def relative_steer(x: torch.Tensor, y: torch.Tensor, v: torch.Tensor, strength: float) -> torch.Tensor:
    """h = x + y; returns h + strength * |h| * v."""
    out = torch.empty_like(x)
    _relative_steer_kernel[(x.shape[0],)](x, y, v, out, strength, x.shape[1],
                                          BLOCK=triton.next_power_of_2(x.shape[1]))
    return out


@relative_steer.register_fake
def _(x, y, v, strength):
    return torch.empty_like(x)


@relative_steer.register_kernel("cpu")
def _(x, y, v, strength):
    h = x.float() + y.float()
    return (h + strength * h.norm(dim=-1, keepdim=True) * v.float()).to(x.dtype)


@register_flop_formula(torch.ops.tutorial.relative_steer)
def _(x_shape, *args, **kwargs) -> int:
    return 5 * math.prod(x_shape)

## Step 2: Check GPU vs. CPU Reference Standalone

Before putting the operator into the inference engine, let's verify one basic question: in plain PyTorch, do the GPU kernel and its CPU reference produce matching results?

This step does not involve pLLM. We generate random `bfloat16` tensors of 8 rows by width 896 (`Qwen2.5-0.5B`'s hidden size), set `strength=0.3`, and invoke `relative_steer` on both GPU and CPU:

In [3]:
x = torch.randn(8, 896, device="cuda", dtype=torch.bfloat16)
y = torch.randn_like(x)
v = torch.nn.functional.normalize(torch.randn(896, device="cuda"), dim=0).bfloat16()
on_gpu = relative_steer(x, y, v, 0.3)
on_cpu = relative_steer(x.cpu(), y.cpu(), v.cpu(), 0.3)
print("GPU 与 CPU 参考的最大差:", (on_gpu.cpu().float() - on_cpu.float()).abs().max().item())

GPU 与 CPU 参考的最大差: 0.0


The output shows `0.0` max difference between GPU and CPU reference, confirming agreement in `bfloat16`.

## Step 3: Prepare the Steering Direction

Next, we compute the layer-12 steering direction vector and save it to disk for the workload to load.

The procedure matches Tutorial 00: take the difference of last-token activations at layer 12 across 5 opposite-meaning phrase pairs and average them. Whereas Tutorial 00 read activations from the vLLM engine, here we run Hugging Face on CPU in a few seconds so Steps 5 and 6 (`explain` and CPU contract checks, along with the direction file they load) can run even on a machine without a GPU. After averaging, we normalize by its norm so `direction` is a unit vector, leaving scaling to the residual stream's own norm $|h|$.

Finally, we save the vector to a safetensors file and load it into an `InstanceState` via `init=pllm.Pretrained(...)`. Why a file instead of Tutorial 00's `StateImport`? Because contract checks run with that actual loaded state: if the direction were only injected externally at `generate` time, it would be zero during contract checks and a bug in the kernel would go unnoticed (as we demonstrate in Step 10). With a `pllm.Pretrained` file, the checks run with the real non-zero direction.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
LAYER = 12
pairs = [("I love", "I hate"), ("This is wonderful", "This is terrible"), ("I feel happy", "I feel sad"),
         ("What a great day", "What an awful day"), ("Everything is amazing", "Everything is horrible")]
hf = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)  # Runs on CPU in a few seconds


@torch.no_grad()
def last_token_activation(text: str) -> torch.Tensor:
    ids = tokenizer(text, return_tensors="pt")
    return hf(**ids, output_hidden_states=True).hidden_states[LAYER + 1][0, -1].float()


vector = torch.stack([last_token_activation(a) - last_token_activation(b) for a, b in pairs]).mean(0)
direction = vector / vector.norm()  # Unit vector; magnitude is set by the residual stream's own norm
print("CAA 向量范数", round(vector.norm().item(), 2),
      "；第 12 层输出的残差流范数（正面短语的最后一个 token，平均）",
      round(torch.stack([last_token_activation(a).norm() for a, _ in pairs]).mean().item(), 2))
DIRECTION = os.path.join(tempfile.gettempdir(), "relative_steer_direction.safetensors")
save_file({"direction": direction.bfloat16()}, DIRECTION)
del hf

CAA 向量范数 7.08 ；第 12 层输出的残差流范数（正面短语的最后一个 token，平均） 18.18


The computed CAA vector has norm `7.08`, and the mean layer-12 residual stream norm (last token of positive phrases) is `18.18`. Thus `strength=0.3` corresponds to adding a vector of norm roughly `5.5` (estimated as $0.3 \times 18.18$).

## Step 4: Write the Workload

Next: how do we wire `relative_steer` into a `pllm.Workload`?

As introduced in Example 07, `@pllm.custom_residual(site="mlp", layers=(12,))` takes over the residual update after layer 12's MLP: `x` is the residual stream entering the MLP, `y` is the MLP output, and the method returns the new residual stream. Here, `add`'s body is simply a direct call to `relative_steer(x, y, v, self.strength)`, just like any Python function.

`strength` is a field on the frozen dataclass, with `strength=0.0` representing no steering:

In [5]:
@dataclass(frozen=True)
class RelativeSteering(pllm.Workload):
    strength: float = 0.0
    path: str = DIRECTION

    @property
    def steer(self) -> pllm.InstanceState:
        return pllm.InstanceState({"direction": lambda m: (m.hidden_size,)}, init=pllm.Pretrained(self.path))

    @pllm.custom_residual(site="mlp", layers=(12,))
    def add(self, x: torch.Tensor, y: torch.Tensor,
            v: torch.Tensor = pllm.parameter("steer", "direction")) -> torch.Tensor:
        return relative_steer(x, y, v, self.strength)

## Step 5: Inspect `explain` Before Launch

Before starting the engine and loading model weights, let's run `pllm.explain` on CPU to see how pLLM analyzes this custom operator. `explain` prints many lines; below we filter to the lines relevant to the operator:

In [6]:
report = pllm.explain(RelativeSteering(0.3), CONFIG)
for line in str(report).splitlines():
    if line.startswith(("forward", "member", "add ", "op ")):
        print(line)

forward  CUDA graphs: no effect needs eager
member  effect  anchor / lane / graph  state reads (origin, distance)
add  custom_residual  model site or inactive method
op       tutorial::relative_steer: add; 4480 [declared] FLOP/row; 5824 logical bytes/row [lower bound]; CPU reference=True, autograd=False; host syncs unverified (fake only)


Let's read the `explain` output line by line:
- `forward  CUDA graphs: no effect needs eager`: the model stays in CUDA graphs, and our kernel will be captured right alongside it.
- `add  custom_residual  model site or inactive method`: `add` is a `custom_residual` method attached at a model site.
- The `op` line lists details for `tutorial::relative_steer`:
  - called from `add`;
  - `4480 [declared] FLOP/row`: 4480 FLOPs per row, marked `[declared]` from our registered formula ($896 \times 5 = 4480$);
  - `5824 logical bytes/row [lower bound]`: at least 5824 logical bytes read/written per row;
  - `CPU reference=True, autograd=False`: a CPU reference is registered, and no backward formula is registered (none needed since we are not training);
  - `host syncs unverified (fake only)`: running `explain` on CPU only sees the `register_fake` stub and cannot inspect whether the GPU kernel performs a GPU-to-CPU sync; that check runs on GPU in Step 9.

## Step 6: CPU Contract Check

Next, we run `testing.check` on CPU. It drives three requests across several fixed schedules and compares results using our registered CPU reference. Passing `neutral=RelativeSteering(0.0)` verifies that `strength=0.0` leaves behavior unchanged:

In [7]:
print(testing.check(RelativeSteering(0.3), CONFIG, neutral=RelativeSteering(0.0)))

status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    custom-residual method math / CPU schedules
PASS    neutral
SKIP    model-site execution: CPU decoder does not execute residual hooks, overlays, spans or PositionState


In the check output, `admission`, `compile`, `custom-residual method math / CPU schedules`, and `neutral` (`strength=0.0` does not alter sampling) all `PASS`. The final line is `SKIP` for a clear reason: the small CPU decoder does not execute `residual hooks`, so real model-site execution must be verified on GPU.

## Step 7: Generation

Now we launch the real engine and test greedy decoding for 40 tokens on two prompts at strengths `0.0`, `+0.3`, and `-0.3`, printing `eager_reason` on each run:

In [8]:
questions = ["Write one sentence about my new job.", "Describe your weekend in one sentence."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=40, temperature=0.0)


def answers(llm) -> list[str]:
    return [output.outputs[0].text for output in llm.generate(prompts, greedy, use_tqdm=False)]


for strength in (0.0, 0.3, -0.3):
    with pllm.make_llm(CONFIG, workload=RelativeSteering(strength)) as llm:
        texts = answers(llm)
        print(f"强度 {strength:+.1f}，eager 原因 {llm.diagnostics().eager_reason!r}")
        for text in texts:
            print("   ", repr(text))

强度 +0.0，eager 原因 ''
    'As an AI language model, I am here to assist you with your queries and provide you with information and insights on various topics. I am always here to help you with your new job.'
    "As an AI language model, I don't have a physical presence, so I don't have a weekend. However, I'm always here to assist you with any questions or tasks you may have!"


强度 +0.3，eager 原因 ''
    'In just a few years, I am excited to bring together the diverse and passionate team of creators and partners to create and share incredible experiences that inspire and inspire. Together, we are creating a world where'
    'My weekend is filled with a wide range of activities that bring me joy and excitement. From exploring the world of science and technology, to exploring the world of art and design, to exploring the world of'


强度 -0.3，eager 原因 ''
    "I'm sorry, but I don't know what kind of job you're trying to describe. I'm a language model and don't have access to your personal information or the specifics of your job."
    "I'm unable to describe my weekend in a one-sentence manner as I am a language model and do not have personal experiences or a subjective description of my behavior. My primary function is to provide informative"


Two things stand out in the output:
1. `eager_reason` is the empty string `''` across all three runs—both the model and our Triton kernel run inside CUDA graphs.
2. At strength `+0.3`, both answers become enthusiastic (`"I am excited to bring together..."`, `"My weekend is filled with a wide range of activities that bring me joy and excitement..."`), while at strength `-0.3` they turn into refusals (`"I'm sorry, but I don't know what kind of job..."`, `"I'm unable to describe my weekend in a one-sentence manner..."`). Again, this direction is just a CAA vector from 5 phrase pairs; changes at non-zero strength only show the intervention works, not that the model improved.

Note also that `strength=+0.0` reads similarly to the unsteered model, though it need not match default native vLLM verbatim: `custom_residual` makes vLLM capture the model's un-fused operators inside CUDA graphs (without inductor fusion), slightly altering floating-point rounding relative to default native vLLM. Bit-for-bit verification is handled by `check_on_gpu` in Step 9, which launches native vLLM with the matching graph setup.

## Step 8: Throughput

Next: does writing this step as a Triton kernel run any faster end-to-end than writing it with standard PyTorch ops?

The 0.5B model sufficed above, whereas for throughput we switch to `Qwen/Qwen2.5-7B-Instruct`: a decode step on 0.5B takes only a few milliseconds, magnifying any fixed overhead, while 7B reflects typical deployment scale (~15 GiB bf16 weights, needing a 24 GiB GPU). Since benchmarking only measures compute, we use a random unit vector of 7B's hidden size (3584) at `strength=0.3`. We run 128 concurrent requests (8 prompts × 16 samples × 256 tokens) with `pllm.benchmark.throughput` (1 warmup round, median of 3 timed rounds, timing full `generate` including prefill) across three variants: native vLLM, Triton-based `RelativeSteering(0.3)`, and `RelativeSteeringTorch(0.3)` implementing the same math with standard PyTorch ops:

In [9]:
@dataclass(frozen=True)
class RelativeSteeringTorch(RelativeSteering):
    """Same math written with standard PyTorch ops as a baseline."""

    @pllm.custom_residual(site="mlp", layers=(12,))
    def add(self, x: torch.Tensor, y: torch.Tensor,
            v: torch.Tensor = pllm.parameter("steer", "direction")) -> torch.Tensor:
        h = x.float() + y.float()
        return (h + self.strength * h.norm(dim=-1, keepdim=True) * v.float()).to(x.dtype)


# On 7B, the rewritten model graph needs ~84 MiB capture pool, exceeding the default 64 MiB (when insufficient, pLLM errors with the required byte count).
BIG = pllm.ModelConfig("Qwen/Qwen2.5-7B-Instruct", max_model_len=512, max_num_seqs=128, max_num_batched_tokens=512,
                       gpu_memory_utilization=0.85, seed=0,
                       execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=128 << 20))
BIG_DIRECTION = os.path.join(tempfile.gettempdir(), "relative_steer_direction_7b.safetensors")
save_file({"direction": torch.nn.functional.normalize(torch.randn(3584), dim=0).bfloat16()}, BIG_DIRECTION)
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=16, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)

rates = {}
with pllm.make_llm(BIG) as plain:
    rates["原生 vLLM"] = throughput(plain, many, sampling)
for name, workload in (("Triton 算子", RelativeSteering(0.3, BIG_DIRECTION)),
                       ("PyTorch 算子", RelativeSteeringTorch(0.3, BIG_DIRECTION))):
    with pllm.make_llm(BIG, workload=workload) as llm:
        rates[name] = throughput(llm, many, sampling)
for name, rate in rates.items():
    print(f"{name:10s}: {rate:6.0f} token/s，为原生 vLLM 的 {rate / rates['原生 vLLM']:.0%}")

原生 vLLM   :   5872 token/s，为原生 vLLM 的 100%
Triton 算子 :   5760 token/s，为原生 vLLM 的 98%
PyTorch 算子:   5755 token/s，为原生 vLLM 的 98%


In this run (RTX 4090, vLLM 0.32.0, `Qwen2.5-7B-Instruct` bf16, CUDA graphs; 128 concurrent requests = 8 × 16, 256 tokens each; median of 3 rounds after 1 warmup):
- Native vLLM: `5872 token/s` (`100%`)
- Triton op: `5760 token/s` (`98%` of native vLLM)
- PyTorch ops: `5755 token/s` (`98%` of native vLLM)

The Triton and PyTorch variants perform virtually identically, well within measurement noise: this kernel brings no speedup here because a single addition at layer 12 is not an end-to-end bottleneck. As stated at the beginning, the reason to write a custom kernel is when your algorithm truly has a step that PyTorch cannot express or runs too slowly.

## Step 9: GPU Contract Check

Earlier, Step 5's `explain` left `host syncs unverified (fake only)`, and Step 6's CPU check skipped model-site execution. Now we run the full contract suite on GPU with `testing.check_on_gpu`.

`check_on_gpu` loads the real engine and verifies:
1. `device equivalence`: replays every method's inputs from the CPU contract check on GPU through the Triton kernel and compares against the CPU reference outputs (within the default floating-point tolerance for the dtype).
2. `host syncs`: checks that no GPU-to-CPU synchronization occurs across eight steady decode steps.
3. `preempted`: forces KV preemption and recomputation, verifying identical results.
4. `neutral`: verifies that `strength=0.0` produces bit-for-bit identical tokens to native vLLM under the same graph configuration.

Because `check_on_gpu` launches multiple engine instances, pLLM recommends running it in a separate process; in this notebook we place it at the end:

In [10]:
print(testing.check_on_gpu(RelativeSteering(0.3), CONFIG, neutral=RelativeSteering(0.0)))

status  contract / reason
PASS    device equivalence: CPU check script method outputs replayed on CUDA
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: greedy and seeded sampling bitwise


All five checks `PASS`: device equivalence holds, eight steady decode steps incur zero host syncs, preemption recomputation matches, and `neutral` is bit-for-bit identical.

In [11]:
@torch.library.custom_op("tutorial::relative_steer_wrong", mutates_args=(), device_types="cuda")
def relative_steer_wrong(x: torch.Tensor, y: torch.Tensor, v: torch.Tensor, strength: float) -> torch.Tensor:
    out = torch.empty_like(x)
    _relative_steer_kernel[(x.shape[0],)](x, y, v, out, strength, x.shape[1],
                                          BLOCK=triton.next_power_of_2(x.shape[1]))
    return out


@relative_steer_wrong.register_fake
def _(x, y, v, strength):
    return torch.empty_like(x)


@relative_steer_wrong.register_kernel("cpu")
def _(x, y, v, strength):
    h = x.float() + y.float()
    return (h + strength * v.float()).to(x.dtype)  # Bug: forgot to multiply by |h|


@dataclass(frozen=True)
class WrongReference(RelativeSteering):
    @pllm.custom_residual(site="mlp", layers=(12,))
    def add(self, x: torch.Tensor, y: torch.Tensor,
            v: torch.Tensor = pllm.parameter("steer", "direction")) -> torch.Tensor:
        return relative_steer_wrong(x, y, v, self.strength)


try:
    print(testing.check_on_gpu(WrongReference(0.3), CONFIG))
except testing.ContractError as error:
    print(error)

1 contract failure(s)
[device equivalence] [device equivalence] add: Tensor-likes are not close!

Mismatched elements: 12715 / 13440 (94.6%)
Greatest absolute difference: 2.53125 at index (5, 581) (up to 1e-05 allowed)
Greatest relative difference: 5536.0 at index (8, 688) (up to 0.016 allowed)


`check_on_gpu` raised `pllm.testing.ContractError`, reporting 1 contract failure: `[device equivalence] add: Tensor-likes are not close!`. Across 13,440 elements, 12,715 (94.6%) exceeded `bfloat16` tolerance, with a greatest absolute difference of `2.53125` (allowed `1e-05`) and greatest relative difference of `5536.0` (allowed `0.016`). The only reason we caught the missing `|h|` factor directly here is that Step 3 loaded a non-zero steering direction via `pllm.Pretrained`, giving the check script real non-zero inputs.

## Summary and Next Steps

Recapping this example:
- Integrating a custom Triton kernel into pLLM requires only wrapping it with PyTorch's `torch.library.custom_op` and calling the op object directly inside workload methods—no pLLM-specific operator API is needed.
- `register_fake` is required (for memory planning and sync checks); `register_kernel("cpu")` is strongly recommended, as without it CPU contract checks cannot execute that math and device-equivalence verification cannot run.
- Pre-launch `pllm.explain` reports the operator's FLOPs, logical read/write bytes, and a fingerprint derived from the registered functions and kernel source.
- `testing.check_on_gpu`'s device-equivalence check catches mismatches between your GPU kernel and CPU reference on replayed real inputs; loading op-dependent state via `pllm.Pretrained` ensures those checks exercise real non-zero values.

Next: in [10 Custom Attention](10_custom_attention.ipynb), we write Quest-style page scoring in PyTorch and show why achieving real speedups requires a custom kernel like the one wired up here.